# Tracking + Pose Estimation — Combined Evaluation

End-to-end notebook running detection → tracking → pose estimation together,
with rich skeleton+box visualisation using `PipelineVisualiser`.

**When to use this notebook:**
- Validating the detect → track → pose pipeline on a new clip
- Tuning ByteTrack parameters (IOU threshold, min confidence)
- Checking skeleton quality before running metric extractors
- Turning pose on/off to compare speed vs quality

**Toggles:**
```python
ENABLE_POSE = True    # set False to run detect+track only
ENABLE_OCR  = False   # set True to also read bib numbers during run
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Config ───────────────────────────────────────────────────────────────────
VIDEO_PATH   = Path('../data/raw_footage/sample_clip.mp4')
TARGET_FPS   = 15
MAX_FRAMES   = 45      # keep small for interactive use; None = full clip
ENABLE_POSE  = True    # ← toggle pose estimation on/off
ENABLE_OCR   = False   # ← toggle bib OCR on/off
CONF_DET     = 0.40    # detection confidence threshold
JOB_ID       = 'notebook-track-pose'

print(f"Video: {VIDEO_PATH}  exists={VIDEO_PATH.exists()}")
print(f"ENABLE_POSE={ENABLE_POSE}  ENABLE_OCR={ENABLE_OCR}")

In [ ]:
from pipeline.ingest import extract_frames

frames, frame_indices, timestamps = [], [], []
for fi, frame, ts in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS):
    frames.append(frame); frame_indices.append(fi); timestamps.append(ts)
    if MAX_FRAMES and len(frames) >= MAX_FRAMES: break

print(f"Extracted {len(frames)} frames  ({timestamps[-1]:.1f}s)  "
      f"resolution {frames[0].shape[1]}×{frames[0].shape[0]}")

In [ ]:
# ── Detection ────────────────────────────────────────────────────────────────
from pipeline.detect import PersonDetector

detector = PersonDetector(conf_threshold=CONF_DET)
all_detections = []
for i, frame in enumerate(frames):
    dets = detector.detect(frame)
    for d in dets: d.frame_idx = frame_indices[i]
    all_detections.append(dets)

total_dets = sum(len(d) for d in all_detections)
print(f"Detection done — {total_dets} detections across {len(frames)} frames")

In [ ]:
# ── Tracking ─────────────────────────────────────────────────────────────────
from pipeline.track import PersonTracker

tracker = PersonTracker()
all_tracks = []
for i, (frame, dets) in enumerate(zip(frames, all_detections)):
    tracks = tracker.update(dets, frame_idx=frame_indices[i])
    all_tracks.append(tracks)

unique_ids = set(t.track_id for ft in all_tracks for t in ft)
print(f"Tracking done — {len(unique_ids)} unique track IDs: {sorted(unique_ids)}")

In [ ]:
# ── Pose estimation (conditional) ────────────────────────────────────────────
if ENABLE_POSE:
    from pipeline.pose import PoseEstimator
    estimator = PoseEstimator()
    all_poses = []
    for i, (frame, tracks) in enumerate(zip(frames, all_tracks)):
        poses = estimator.estimate_batch(frame, tracks)
        all_poses.append(poses)
        if i % 10 == 0:
            print(f"  frame {i:3d} / {len(frames)}  poses={len(poses)}")
    print(f"\nPose estimation done — {sum(len(p) for p in all_poses)} total poses")
else:
    all_poses = [[] for _ in frames]
    print("Pose estimation SKIPPED (ENABLE_POSE=False)")

In [ ]:
# ── OCR (conditional) ────────────────────────────────────────────────────────
if ENABLE_OCR:
    from pipeline.ocr import BibOCR, resolve_bibs
    ocr = BibOCR()
    frame_readings = []
    for i, (frame, tracks) in enumerate(zip(frames, all_tracks)):
        if i % 5 == 0:
            readings = ocr.read_frame(frame, tracks)
            frame_readings.append(readings)
    resolved = resolve_bibs(frame_readings)
    for frame_tracks in all_tracks:
        for track in frame_tracks:
            bib, conf = resolved.get(track.track_id, (None, 0.0))
            track.bib_number = bib; track.bib_confidence = conf
    print(f"OCR done. Resolved bibs: {resolved}")
else:
    resolved = {}
    print("OCR SKIPPED (ENABLE_OCR=False) — bibs shown as unresolved")

In [ ]:
# ── Calibration (single-axis fallback) ───────────────────────────────────────
from pipeline.calibrate import Calibrator
calibrator  = Calibrator()
calibration = calibrator.calibrate_single_axis(frames[0])
print(f"Calibration: {calibration.method}  valid={calibration.is_valid}  "
      f"px/cm={calibration.pixels_per_cm}")

In [ ]:
# ── Visualise using PipelineVisualiser.annotate_frame (static) ───────────────
from pipeline.visualise import PipelineVisualiser, VisOptions

opts = VisOptions(
    show_boxes=True,
    show_skeleton=ENABLE_POSE,   # hide skeleton layer if pose was skipped
    show_calibration_grid=calibration.is_valid,
    show_hud=False,              # no test results yet
    show_test_overlay=False,
    show_flags=True,
    show_frame_counter=True,
)

SAMPLE_EVERY = max(1, len(frames) // 9)
sample_idx   = list(range(0, len(frames), SAMPLE_EVERY))[:9]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, fi in zip(axes.flat, sample_idx):
    canvas = PipelineVisualiser.annotate_frame(
        frame=frames[fi],
        tracks=all_tracks[fi],
        poses=all_poses[fi],
        calibration=calibration,
        results=[],
        test_type='explosiveness',
        frame_idx=frame_indices[fi],
        opts=opts,
    )
    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.set_title(f"t={timestamps[fi]:.1f}s  tracks={len(all_tracks[fi])}", fontsize=9)
    ax.axis('off')

label = "Tracking + Pose" if ENABLE_POSE else "Tracking only"
plt.suptitle(label, fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Save annotated video ─────────────────────────────────────────────────────
from pathlib import Path

OUTPUT_PATH = Path('../data/annotated/tracking_pose_eval.mp4')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with PipelineVisualiser(OUTPUT_PATH, test_type='explosiveness',
                        fps=TARGET_FPS, options=opts) as vis:
    for frame, tracks, poses, ts in zip(frames, all_tracks, all_poses, timestamps):
        vis.write_frame(frame, tracks, poses, calibration, [], timestamp_s=ts)

print(f"Saved annotated video → {OUTPUT_PATH}")

In [ ]:
# ── Save everything to cache ─────────────────────────────────────────────────
from pipeline.cache import PipelineCache

cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))
cache.save_ingest(frame_indices, timestamps)
cache.save_detections(all_detections)
cache.save_tracks(all_tracks)
if ENABLE_POSE:
    cache.save_poses(all_poses)

print(f"Cached as job_id='{JOB_ID}'")
print(cache.summary())